In [1]:
# 1. Install necessary libraries
!pip install spacy pandas sentence-transformers
!python -m spacy download en_core_web_sm

import spacy
import pandas as pd
import re
from tqdm import tqdm # This adds a progress bar for large data

# Load the spaCy engine
nlp = spacy.load("en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Load a public dataset directly into Colab
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.DataFrame(pd.read_csv(url))

# Let's take a smaller sample (5,000) to keep the 30-min timeline
df = df.sample(5000).reset_index(drop=True)

print(f"Dataset Loaded. Total reviews to process: {len(df)}")
print(df.head(3))

Dataset Loaded. Total reviews to process: 5000
                                              review sentiment
0  Not a movie, but a lip synched collection of p...  negative
1  I don't see why everyone is bombing this so mu...  positive
2  I wouldn't dare say this film is better than t...  positive


In [3]:
class TextCleaner:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

    def clean(self, text):
        # 1. Remove HTML tags (common in web data)
        text = re.sub(r'<.*?>', '', text)
        # 2. Lowercase and remove special characters
        text = re.sub(r'[^a-zA-Z\s]', '', text).lower()

        # 3. Process with spaCy for Lemmatization and Stop Word removal
        doc = self.nlp(text)
        # Your notes mention "Stop Words"—we filter them here:
        tokens = [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 2]

        return " ".join(tokens)

# Initialize and apply to our dataset
cleaner = TextCleaner()
tqdm.pandas() # Connect progress bar to pandas
df['cleaned_review'] = df['review'].progress_apply(cleaner.clean)

print("\n--- Phase 1 Complete ---")
print(f"Original: {df['review'][0][:70]}...")
print(f"Cleaned:  {df['cleaned_review'][0][:70]}...")

100%|██████████| 5000/5000 [01:39<00:00, 50.12it/s]


--- Phase 1 Complete ---
Original: Not a movie, but a lip synched collection of performances from acts th...
Cleaned:  movie lip synche collection performance act british invasion follow dy...


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained "Semantic" model
# This replaces the 'One-Hot' problem you noted with 'Dense Vectors'
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Converting text to vectors (this is the math input)...")
# Cnonvert our cleaed reviews into a matrix of numbers (X)
X = embedder.encode(df['cleaned_review'].tolist(), show_progress_bar=True)

# Convert labels (positive/negative) into numbers (y)
# Positive = 1, Negative = 0
y = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values

print(f"Input Shape: {X.shape}") # Should be (5000, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Converting text to vectors (this is the math input)...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Input Shape: (5000, 384)


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SentiraNet(nn.Module):
    def __init__(self, input_size=384, hidden_size=128):
        super(SentiraNet, self).__init__()
        # Layer 1: The 'w' weights from your notes connecting Input to Hidden
        self.fc1 = nn.Linear(input_size, hidden_size)

        # Layer 2: Hidden to Output (2 classes: Pos/Neg)
        self.fc2 = nn.Linear(hidden_size, 2)

    def forward(self, x):
        # Apply ReLU activation to the first layer
        # (This helps avoid the "Zero Slope" problem you noted)
        x = F.relu(self.fc1(x))

        # Apply Softmax to the output to get probabilities (Page 3 of notes)
        x = self.fc2(x)
        return x # PyTorch's CrossEntropyLoss handles the Softmax internally

# Initialize the model
model = SentiraNet()
print(model)

SentiraNet(
  (fc1): Linear(in_features=384, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [15]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. SETUP DATA (The 'Inputs' to your Neural Net)
# Ensure X and y from Phase 2 are converted to PyTorch format
train_xt, test_xt, train_yt, test_yt = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = TensorDataset(torch.FloatTensor(train_xt), torch.LongTensor(train_yt))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. SETUP THE BRAIN (The Weights 'w' and Bias 'b')
model = SentiraNet() # Re-initializing the model
criterion = nn.CrossEntropyLoss() # Measuring the 'Error'

# This is the Momentum logic from your notes!
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. THE TRAINING LOOP (The 'Delta Rule' cycle)
print("Starting Training... Watch the Loss go down!")
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        # A. Clear old gradients (zero out the 'slope')
        optimizer.zero_grad()

        # B. Forward Pass (Make a guess)
        output = model(data)

        # C. Calculate Error (E)
        loss = criterion(output, target)

        # D. Backpropagation (Calculate the Gradient/Delta)
        loss.backward()

        # E. Update Weights (The step function from your notes)
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] complete. Average Loss: {avg_loss:.4f}")

print("\nSuccess! Your model has learned from the data.")

Starting Training... Watch the Loss go down!
Epoch [1/10] complete. Average Loss: 0.5771
Epoch [2/10] complete. Average Loss: 0.4618
Epoch [3/10] complete. Average Loss: 0.4360
Epoch [4/10] complete. Average Loss: 0.4269
Epoch [5/10] complete. Average Loss: 0.4151
Epoch [6/10] complete. Average Loss: 0.4132
Epoch [7/10] complete. Average Loss: 0.4068
Epoch [8/10] complete. Average Loss: 0.3943
Epoch [9/10] complete. Average Loss: 0.3906
Epoch [10/10] complete. Average Loss: 0.3898

Success! Your model has learned from the data.


In [17]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for data, target in test_loader:
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

accuracy = 100 * correct / total
print(f"Final Accuracy on Unseen Data: {accuracy:.2f}%")

Final Accuracy on Unseen Data: 81.30%


In [20]:
def start_sentiment_bot():
    print("--- 🤖 Sentira-Ops: AI Sentiment Analysis Bot 🤖 ---")
    print("Type 'exit' or 'quit' to stop the program.\n")

    model.eval() # Set to evaluation mode

    while True:
        # 1. Ask for User Input
        user_input = input("Enter a movie review to analyze: ")

        # Check if user wants to quit
        if user_input.lower() in ['exit', 'quit']:
            print("Goodbye! Thanks for testing.")
            break

        if not user_input.strip():
            continue

        # 2. Pre-process and Vectorize (The Phase 1 & 2 logic)
        with torch.no_grad():
            cleaned = cleaner.clean(user_input)
            vector = torch.FloatTensor(embedder.encode([cleaned]))

            # 3. Forward Pass (The Phase 3 'Brain' logic)
            output = model(vector)

            # 4. Calculate Probabilities (Softmax from your notes)
            probs = torch.softmax(output, dim=1)
            prediction = torch.argmax(probs).item()
            confidence = probs[0][prediction].item()

            # 5. Output Result
            label = "POSITIVE ✅" if prediction == 1 else "NEGATIVE ❌"
            print(f"Result: {label}")
            print(f"Confidence: {confidence*100:.2f}%")
            print("-" * 40)

# Start the interactive session
start_sentiment_bot()

--- 🤖 Sentira-Ops: AI Sentiment Analysis Bot 🤖 ---
Type 'exit' or 'quit' to stop the program.

Enter a movie review to analyze: I am very much in affection of this movie
Result: POSITIVE ✅
Confidence: 94.37%
----------------------------------------
Enter a movie review to analyze: exit
Goodbye! Thanks for testing.


In [21]:
# Save the model's 'Brain' (State Dict)
torch.save(model.state_dict(), 'sentira_model.pth')
print("Model saved as 'sentira_model.pth'. Download this from the Colab sidebar for your GitHub!")

Model saved as 'sentira_model.pth'. Download this from the Colab sidebar for your GitHub!
